In [0]:
# COMMAND ----------
# STEP 1: INITIALIZE LIBRARIES & DEFINE ENVIRONMENTAL CONFIGURATIONS
# COMMAND ----------
from pyspark.sql.functions import current_timestamp

# Clean, version-controlled paths to your Unity Catalog Volume and Schema
VOLUME_PATH = "/Volumes/dev_catalog/customer_analytics/landing"
TARGET_SCHEMA = "dev_catalog.customer_analytics"


In [0]:
from pyspark.sql.functions import col, current_timestamp

raw_customers_df = (
    spark.read
    .format("parquet")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOLUME_PATH}/Customers/")
    # CRITICAL METADATA: Track execution time and the source file name
    .withColumn("bronze_ingestion_time", current_timestamp())
    # 🌟 FIX: Use the Unity Catalog approved hidden metadata column
    .withColumn("source_file_name", col("_metadata.file_path"))
)

# 2. Append safely to the historical ledger
(
    raw_customers_df.write
    .format("delta")
    .mode("append")  # 👈 Changes from 'overwrite' to 'append'
    .saveAsTable(f"{TARGET_SCHEMA}.bronze_customers")
)

print(f"✅ Orders data appended successfully. Total historical row count: {spark.table(f'{TARGET_SCHEMA}.bronze_customers').count()}")

In [0]:
# COMMAND ----------
# STEP 3: INGEST ORDERS VIA BATCH LOAD (APPEND-ONLY HISTORICAL LEDGER)
# COMMAND ----------
from pyspark.sql.functions import current_timestamp, input_file_name



# 1. Read files directly from the Volume path
raw_orders_df = (
    spark.read
    .format("parquet")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOLUME_PATH}/Orders/")
    # Add metadata for production auditing
    .withColumn("bronze_ingestion_time", current_timestamp())
    .withColumn("source_file_name", col("_metadata.file_path"))
)

# 2. Append directly to your Managed Unity Catalog Table
(
    raw_orders_df.write
    .format("delta")
    .mode("append")  # Keeps every historic record safe
    .saveAsTable(f"{TARGET_SCHEMA}.bronze_orders")
)

print(f"✅ Orders data appended successfully. Total historical row count: {spark.table(f'{TARGET_SCHEMA}.bronze_orders').count()}")
